# Week 1 - Data Workflow & Acquisition

**Topics:** DS lifecycle • project structure • reproducibility • CSV/Excel acquisition

## 0. Setup

In [1]:
import os
import sys
import platform
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('pandas:', pd.__version__)

Python: 3.13.7
Platform: Linux-6.17.0-19-generic-x86_64-with-glibc2.42
pandas: 3.0.1


## 1. The DS Lifecycle (Workflow Map)
A typical (iterative) lifecycle:
1. **Problem framing**: goal, scope, success metric
2. **Data acquisition**: sources, permissions, extraction
3. **Data understanding**: schema, missingness, anomalies
4. **Data cleaning & feature engineering**
5. **Modeling / analysis**
6. **Evaluation**: metrics, validation, error analysis
7. **Communication & deployment**: dashboards, reports, pipelines
8. **Monitoring & maintenance**

**Problem Statement:** Predicting hospital appointment no-shows.

**Success Metric:** F1-score for predicting where "NoShow = yes".

## 2. Project Structure (Repeatable & Collaborative)

```
project/
  README.md
  data/
    raw/        # immutable source files
    interim/    # intermediate outputs
    processed/  # analysis-ready
  notebooks/    # exploration, teaching
  src/          # reusable functions/modules
  reports/      # figures, tables, exports
  configs/      # settings, paths
  tests/        # unit/data tests
```

In [5]:
from pathlib import Path

PROJECT_ROOT = Path.cwd() / "no_show_predict"
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS = PROJECT_ROOT / "reports"
CONFIGS = PROJECT_ROOT / "configs"

for p in [DATA_RAW, DATA_INTERIM, DATA_PROCESSED, REPORTS, CONFIGS]:
    p.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

PosixPath('/media/irseora/external/Repos/Data-Science/no_show_predict')

In [6]:
readme_text = """# Demo Project: Data Workflow & Acquisition

## Purpose
Teaching example for:
- DS lifecycle
- Project structure
- Reproducibility basics
- Reading CSV/Excel reliably

## How to run
1. Create environment (optional): `python -m venv .venv` then install requirements
2. Run `Assignment1_DataWorkflow.ipynb` cells in order

## Data folders
- `data/raw/`: immutable source files
- `data/interim/`: intermediate outputs
- `data/processed/`: analysis-ready outputs

## Outputs
- `reports/`: tables/figures for sharing
"""

(PROJECT_ROOT / "README.md").write_text(readme_text)
print("Wrote:", PROJECT_ROOT / "README.md")

Wrote: /media/irseora/external/Repos/Data-Science/no_show_predict/README.md


## 3. Reproducibility Essentials

In [7]:
# Freeze a minimal requirements file
requirements = [
    f"pandas=={pd.__version__}",
    f"numpy=={np.__version__}",
    "openpyxl"  # needed for reading/writing .xlsx with pandas
]
(PROJECT_ROOT / "requirements.txt").write_text("\n".join(requirements) + "\n")
print((PROJECT_ROOT / "requirements.txt").read_text())

pandas==3.0.1
numpy==2.4.2
openpyxl



In [8]:
# Control randomness
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Demonstrate determinism
np.random.rand(3)

array([0.37454012, 0.95071431, 0.73199394])

In [9]:
# Configuration file
config = {
    "seed": SEED,
    "raw_csv": "appointments_raw.csv",
    "date_format": "%Y-%m-%d",
    "currency_columns": ["fee"],
    "validation_rules": {
        "age_min": 0,
        "age_max": 120,
        "fee_min": 0,
        "waiting_days_min": 0
    }
}

config_path = CONFIGS / "config.json"
config_path.write_text(json.dumps(config, indent=2))

print("Wrote:", config_path)
print(config_path.read_text())

Wrote: /media/irseora/external/Repos/Data-Science/no_show_predict/configs/config.json
{
  "seed": 42,
  "raw_csv": "appointments_raw.csv",
  "date_format": "%Y-%m-%d",
  "currency_columns": [
    "fee"
  ],
  "validation_rules": {
    "age_min": 0,
    "age_max": 120,
    "fee_min": 0,
    "waiting_days_min": 0
  }
}


## 4. Data Acquisition from CSV

### 4.1. Basic Read

In [10]:
raw_data_path = DATA_RAW / "data_raw.csv"
df_raw = pd.read_csv(raw_data_path)
print(df_raw.shape)
df_raw.head()

(110527, 14)


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


### 4.2. Inspect Schema & Quality

In [11]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 110527 entries, 0 to 110526
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   PatientId       110527 non-null  float64
 1   AppointmentID   110527 non-null  int64  
 2   Gender          110527 non-null  str    
 3   ScheduledDay    110527 non-null  str    
 4   AppointmentDay  110527 non-null  str    
 5   Age             110527 non-null  int64  
 6   Neighbourhood   110527 non-null  str    
 7   Scholarship     110527 non-null  int64  
 8   Hipertension    110527 non-null  int64  
 9   Diabetes        110527 non-null  int64  
 10  Alcoholism      110527 non-null  int64  
 11  Handcap         110527 non-null  int64  
 12  SMS_received    110527 non-null  int64  
 13  No-show         110527 non-null  str    
dtypes: float64(1), int64(8), str(5)
memory usage: 11.8 MB


In [12]:
df_raw.isna().sum().sort_values(ascending=False)

PatientId         0
AppointmentID     0
Gender            0
ScheduledDay      0
AppointmentDay    0
Age               0
Neighbourhood     0
Scholarship       0
Hipertension      0
Diabetes          0
Alcoholism        0
Handcap           0
SMS_received      0
No-show           0
dtype: int64

In [13]:
df_raw.duplicated().sum()

np.int64(0)

### 4.3. Parsing

In [14]:
def parse_data(df_raw):
    df = df_raw.copy()
    
    # Rename for consistency
    df = df.rename(columns={
        "Hipertension": "Hypertension",
        "Handcap": "Handicap",
        "SMS_received": "SMSReceived",
        "No-show": "NoShow"
    })

    # IDs
    df["PatientId"] = df["PatientId"].astype(str)
    df["AppointmentID"] = df["AppointmentID"].astype(str)

    # Dates
    df["ScheduledDay"] = pd.to_datetime(
        df["ScheduledDay"], errors = "coerce",
    )
    df["AppointmentDay"] = pd.to_datetime(
        df["AppointmentDay"], errors = "coerce",
    )

    # Categoricals
    df["Gender"] = df["Gender"].astype("category")
    df["Neighbourhood"] = df["Neighbourhood"].astype("category")

    # Binary -> bool
    binary_cols = [
        "Scholarship", "Hypertension", "Diabetes",
        "Alcoholism", "Handicap", "SMSReceived"
    ]
    for col in binary_cols:
        df[col] = df[col].astype(bool)

    # Target
    df["NoShow"] = df["NoShow"].map({ 
        "Yes": True,
        "No": False
    })
    
    # Derived column: Waiting time in days
    df["WaitingDays"] = (
        df["ScheduledDay"] - df["AppointmentDay"]
    ).dt.days

    return df

df_interim = parse_data(df_raw)
df_interim.info()

# Saving
interim_path = DATA_INTERIM / "appointments_interim.csv"
df_interim.to_csv(interim_path, index = False)

<class 'pandas.DataFrame'>
RangeIndex: 110527 entries, 0 to 110526
Data columns (total 15 columns):
 #   Column          Non-Null Count   Dtype              
---  ------          --------------   -----              
 0   PatientId       110527 non-null  str                
 1   AppointmentID   110527 non-null  str                
 2   Gender          110527 non-null  category           
 3   ScheduledDay    110527 non-null  datetime64[us, UTC]
 4   AppointmentDay  110527 non-null  datetime64[us, UTC]
 5   Age             110527 non-null  int64              
 6   Neighbourhood   110527 non-null  category           
 7   Scholarship     110527 non-null  bool               
 8   Hypertension    110527 non-null  bool               
 9   Diabetes        110527 non-null  bool               
 10  Alcoholism      110527 non-null  bool               
 11  Handicap        110527 non-null  bool               
 12  SMSReceived     110527 non-null  bool               
 13  NoShow          110527 no

### 4.4. Validation & Saving

In [23]:
def validate_data(df_interim):
    df = df_interim.copy()

    # Unrealistic ages
    median_age = df["Age"].median()

    neg_age_mask = df["Age"] < 0
    high_age_mask = df["Age"] > 120

    nr_neg_age = neg_age_mask.sum()
    nr_high_age = high_age_mask.sum()

    df.loc[neg_age_mask, "Age"] = median_age
    df.loc[high_age_mask, "Age"] = median_age

    # Wrong date order
    wrong_date_mask = df["ScheduledDay"] < df["AppointmentDay"]
    nr_wrong_dates = wrong_date_mask.sum()
    
    df.loc[wrong_date_mask, ["AppointmentDay", "ScheduledDay"]] = \
        df.loc[wrong_date_mask, ["ScheduledDay", "AppointmentDay"]].values
    df["WaitingDays"] = (
        df["ScheduledDay"] - df["AppointmentDay"]
    ).dt.days

    # Report
    print(f"Negative ages fixed: {nr_neg_age}")
    print(f"Unrealistic ages fixed: {nr_high_age}")
    print(f"Swapped incorrect dates: {nr_wrong_dates}")

    return df

df_processed = validate_data(df_interim)
df_processed.info()

# Saving
processed_path = DATA_PROCESSED / "appointments_processed.csv"
df_processed.to_csv(processed_path, index = False)

Negative ages fixed: 1
Unrealistic ages fixed: 0
Swapped incorrect dates: 71959
<class 'pandas.DataFrame'>
RangeIndex: 110527 entries, 0 to 110526
Data columns (total 15 columns):
 #   Column          Non-Null Count   Dtype              
---  ------          --------------   -----              
 0   PatientId       110527 non-null  str                
 1   AppointmentID   110527 non-null  str                
 2   Gender          110527 non-null  category           
 3   ScheduledDay    110527 non-null  datetime64[us, UTC]
 4   AppointmentDay  110527 non-null  datetime64[us, UTC]
 5   Age             110527 non-null  int64              
 6   Neighbourhood   110527 non-null  category           
 7   Scholarship     110527 non-null  bool               
 8   Hypertension    110527 non-null  bool               
 9   Diabetes        110527 non-null  bool               
 10  Alcoholism      110527 non-null  bool               
 11  Handicap        110527 non-null  bool               
 12  SMS

### 4.5. Summary

In [24]:
summary = pd.DataFrame([
    {"metric": "rows_raw_loaded", "value": int(len(df_raw))},
    {"metric": "rows_interim", "value": int(len(df_interim))},
    {"metric": "rows_processed", "value": int(len(df_processed))},

    {"metric": "unparsed_appointment_date_interim", "value": int(df_interim["AppointmentDate"].isna().sum())},
    {"metric": "unparsed_booking_date_interim", "value": int(df_interim["BookingDate"].isna().sum())},

    {"metric": "duplicate_appointment_id_interim", "value": int(df_interim.duplicated(subset=["AppointmentID"]).sum())},
    {"metric": "duplicate_appointment_id_processed", "value": int(df_processed.duplicated(subset=["AppointmentID"]).sum())},

    {"metric": "missing_patient_age_processed", "value": int(df_processed["Age"].isna().sum())},

    {"metric": "no_show_rate_yes_processed", "value": float((df_processed["NoShow"] == "Yes").mean())},
])

summary_path = REPORTS / "summary_table.csv"
summary.to_csv(summary_path, index=False)
summary

KeyError: 'AppointmentDate'